In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

img_height = 255
img_width = 255
batch_size = 32
AUTOTUNE = tf.data.AUTOTUNE

#(https://www.tensorflow.org/tutorials/load_data/images?hl=ko)
(train_ds, val_ds, test_ds), metadata = tfds.load(
    'tf_flowers',
    split=['train[:80%]','train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

num = 20
def prepare(ds, batch = 1, shuffle=False, augment=False):
    preprocess_input = tf.keras.applications.mobilenet_v3.preprocess_input
    # resize and rescale all datasets
    ds = ds.map(lambda x,y:(tf.image.resize(x,[img_height, img_width]), y), num_parallel_calls=AUTOTUNE)
    
    # batch all datasets
    ds = ds.batch(batch_size)
        
    #prefetch()
    return ds.prefetch(buffer_size=AUTOTUNE)


2025-06-24 12:03:54.364739: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-24 12:03:54.365084: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-24 12:03:54.367029: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-24 12:03:54.372156: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750734234.381123   56486 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750734234.38

In [2]:
num_classes=metadata.features['label'].num_classes
label_name = metadata.features['label'].names
print(label_name, ", classnum : ", num_classes, ", type: ",type(label_name))

test_ds = prepare(test_ds, num)
image_test, label_test = next(iter(test_ds))
image_test = np.array(image_test)
label_test = np.array(label_test, dtype='int')

#모델불러오기
model = tf.keras.models.load_model('transfer_learning_flower_2.keras')
model.summary()

predict = model.predict(image_test)
predicted_classes = np.argmax(predict, axis=1)

2025-06-24 12:03:55.654128: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2025-06-24 12:03:55.682565: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


['dandelion', 'daisy', 'tulips', 'sunflowers', 'roses'] , classnum :  5 , type:  <class 'list'>


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 255, 255, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 8, 8, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         2,885 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,801,793 (10.69 MB)

 Trainable params: 929,893 (3.55 MB)

 Non-trainable params: 12,112 (47.31 KB)

 Optimizer params: 1,859,788 (7.09 MB)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step


In [3]:
print("실제 레이블 | 예측 레이블")
print("-------------------------")
for ll in range((label_test.size)):
    print(label_name[label_test[ll]], "|", label_name[predicted_classes[ll]])

print("-------------------------")

accuracy = np.mean(predicted_classes == label_test)
print(f"정확도 : {accuracy:.2%}")

실제 레이블 | 예측 레이블
-------------------------
roses | daisy
dandelion | dandelion
dandelion | dandelion
tulips | daisy
dandelion | dandelion
dandelion | dandelion
tulips | daisy
daisy | dandelion
sunflowers | sunflowers
dandelion | dandelion
dandelion | dandelion
dandelion | dandelion
sunflowers | sunflowers
roses | sunflowers
dandelion | dandelion
sunflowers | sunflowers
tulips | sunflowers
dandelion | dandelion
tulips | daisy
roses | daisy
tulips | sunflowers
dandelion | dandelion
tulips | daisy
dandelion | dandelion
daisy | daisy
sunflowers | sunflowers
daisy | daisy
roses | sunflowers
roses | daisy
sunflowers | sunflowers
dandelion | dandelion
roses | roses
-------------------------
정확도 : 62.50%
